In [8]:
from pyrocko import util, model, io, trace, moment_tensor, gmtpy,orthodrome
import pyrocko.moment_tensor as pmt
from pyrocko import orthodrome as od
from pyrocko.guts import load

# from seiscloud import plot as scp
# from seiscloud import cluster as scc
import numpy as np
import os
# import shutil
import matplotlib.pyplot as plt

import re
from pathlib import Path
from datetime import datetime

import yaml

# CLASS TO LOAD .YAML FILES AND READ PARAMENTER RESULTS
class IgnoreTagsLoader(yaml.SafeLoader):
    pass

def ignore_unknown(loader, tag_suffix, node):
    if isinstance(node, yaml.MappingNode):
        return loader.construct_mapping(node)
    elif isinstance(node, yaml.SequenceNode):
        return loader.construct_sequence(node)
    else:
        return loader.construct_scalar(node)

IgnoreTagsLoader.add_multi_constructor('!', ignore_unknown)

In [9]:
workdir='../' 
reportdir=os.path.join(workdir,'report')                                

catdir=os.path.join(workdir,'CAT')
catname=os.path.join(catdir,'catalogue_flegrei_VLP.pf')                 # CHANGE
refevents=model.load_events(catname)
mttargets = [ev for ev in refevents]

badmtsols = ['']    # exclude some events
print(f'Catalogue: {catname}')
print('All events in catalogue:', len(mttargets))
goodmttargets = [ev for ev in mttargets if ev.name not in badmtsols]
print('Good events in catalogue:', len(goodmttargets))

# Insert inversions names
inversions= ['cmt_composite_LF_std_']                                    # CHANGE

# Insert source prefixes 
source_prefixes = ['vlp']                                             # CHANGE 'vt' 'vlp'
par_names = ['time','north_shift','east_shift',
              'depth','magnitude',
              'rmnn','rmee','rmdd','rmne','rmnd','rmed',
              'duration','frequency',
              'strike1','dip1','rake1']

print(f'\nInversions selected: {inversions}')
print(f'Sources extracted: {source_prefixes}')

ev_stats={}

# new catalogue name
new_catalogue_name = 'catalogue_flegrei_composite_MT_LF_std'            # CHANGE catalogue_flegrei_composite_MT_LF_std

Catalogue: ../CAT/catalogue_flegrei_VLP.pf
All events in catalogue: 19
Good events in catalogue: 19

Inversions selected: ['cmt_composite_LF_std_']
Sources extracted: ['vlp']


In [10]:
#reportdir=os.path.join(workdir,'report') # pre-defined
#reportdir = '/Users/giaco/UNI/PhD_CODE/GIT/CAMPI_FLEGREI_moment_tensor/report'
reportdir = '/Users/giaco/UNI/PhD_CODE/GIT/Grond_composite_test_results/report'

counter_total_reports = 0
for ev in goodmttargets:
    counter = 0
    for vrs in inversions: # main report
        targetdir = os.path.join(reportdir, ev.name, vrs + ev.name)
        #if not os.path.isdir(targetdir):
            #print(ev.name, 'missing report dir', targetdir)
        if os.path.isdir(targetdir):
            counter += 1
            fname = os.path.join(targetdir, 'stats.yaml')     # results
            if os.path.isfile(fname):
                # Read file
                with open(fname, 'r') as f:
                    data = yaml.load(f, Loader=IgnoreTagsLoader)    # unsing loader class
                
                # acces 'paramenter_stats_list'
                parameter_stats = data['parameter_stats_list']
                #print(parameter_stats)
                for pref in source_prefixes:
                    ev_stats[ev.name]={pref +'.'+ par :0 for par in par_names}
                for p in parameter_stats:
                    if p['name'] in list(ev_stats[ev.name].keys()):
                        #CHANGE: choose values to extract
                        ev_stats[ev.name][p['name']]= {'best' : float( f"{p['best']:.8g}" ),   
                                                        'percentile16' : float( f"{p['percentile16']:.8g}" ),
                                                        'percentile84' : float( f"{p['percentile84']:.8g}" ),
                                                        'mean' : float( f"{p['mean']:.8g}" ),
                                                        'std' : float( f"{p['std']:.8g}" )
                                                        }
                        # print values
                        #print(p['name'], p['best'], p['percentile16'], p['percentile84'])

    if counter == 0:
        print(f'WARNING: {ev.name} does not have any report directories!')
    elif counter > 1:
        print(f'WARNING: {ev.name} has MULTIPLE report directories!')
    else:
        print(f'NICE! {ev.name} has report directory: {targetdir}')
        counter_total_reports += 1

print(f'\nTotal events with report directories: {counter_total_reports} out of {len(goodmttargets)}')

NICE! flegrei_2018_09_18_21_36_41 has report directory: /Users/giaco/UNI/PhD_CODE/GIT/Grond_composite_test_results/report/flegrei_2018_09_18_21_36_41/cmt_composite_LF_std_flegrei_2018_09_18_21_36_41
NICE! flegrei_2023_06_11_06_44_25 has report directory: /Users/giaco/UNI/PhD_CODE/GIT/Grond_composite_test_results/report/flegrei_2023_06_11_06_44_25/cmt_composite_LF_std_flegrei_2023_06_11_06_44_25
NICE! flegrei_2023_09_26_07_10_29 has report directory: /Users/giaco/UNI/PhD_CODE/GIT/Grond_composite_test_results/report/flegrei_2023_09_26_07_10_29/cmt_composite_LF_std_flegrei_2023_09_26_07_10_29
NICE! flegrei_2023_10_02_20_08_26 has report directory: /Users/giaco/UNI/PhD_CODE/GIT/Grond_composite_test_results/report/flegrei_2023_10_02_20_08_26/cmt_composite_LF_std_flegrei_2023_10_02_20_08_26
NICE! flegrei_2024_04_27_03_44_56 has report directory: /Users/giaco/UNI/PhD_CODE/GIT/Grond_composite_test_results/report/flegrei_2024_04_27_03_44_56/cmt_composite_LF_std_flegrei_2024_04_27_03_44_56
NICE!

In [11]:
print(ev_stats)

{'flegrei_2018_09_18_21_36_41': {'vlp.time': {'best': 0.40613639, 'percentile16': -0.23178302, 'percentile84': 0.85810633, 'mean': 0.33128807, 'std': 0.56559629}, 'vlp.north_shift': {'best': 1414.4677, 'percentile16': -803.13892, 'percentile84': 1414.4677, 'mean': 361.28645, 'std': 1147.7957}, 'vlp.east_shift': {'best': -2519.6064, 'percentile16': -4450.4822, 'percentile84': -1148.821, 'mean': -2678.8612, 'std': 1553.5374}, 'vlp.depth': {'best': 4398.6737, 'percentile16': 3388.9834, 'percentile84': 4275.6749, 'mean': 3855.9258, 'std': 397.74414}, 'vlp.magnitude': {'best': 1.1652335, 'percentile16': 1.105043, 'percentile84': 1.3242712, 'mean': 1.2148392, 'std': 0.1124291}, 'vlp.rmnn': {'best': -0.47437388, 'percentile16': -0.65272546, 'percentile84': 0.42194174, 'mean': -0.14190368, 'std': 0.49790266}, 'vlp.rmee': {'best': 0.22362923, 'percentile16': -0.61414795, 'percentile84': 0.67618207, 'mean': 0.032089272, 'std': 0.56347948}, 'vlp.rmdd': {'best': -0.49487487, 'percentile16': -0.762

In [12]:
relocate=True             #CHANGE
update_time=False          #CHANGE
solution = 'best'          #CHANGE w/ 'mean'

events_results = []
for ev in goodmttargets:
    if ev.name in list(ev_stats.keys()) :
        for source_name in source_prefixes:
        # create events objects for sub-problems
            m0=pmt.magnitude_to_moment(ev_stats[ev.name][source_name +'.'+'magnitude'][solution])
            tmp_mt = pmt.MomentTensor(mnn=ev_stats[ev.name][source_name +'.'+'rmnn'][solution] * m0,
                                    mee=ev_stats[ev.name][source_name +'.'+'rmee'][solution] * m0,
                                    mdd=ev_stats[ev.name][source_name +'.'+'rmdd'][solution] * m0,
                                    mne=ev_stats[ev.name][source_name +'.'+'rmne'][solution] * m0,
                                    mnd=ev_stats[ev.name][source_name +'.'+'rmnd'][solution] * m0,
                                    med=ev_stats[ev.name][source_name +'.'+'rmed'][solution] * m0,
                                    moment=m0)

            if relocate:    # re-locate         #CHANGE
                nlat, nlon = od.ne_to_latlon(ev.lat, ev.lon,
                                np.array([ev_stats[ev.name][source_name +'.'+'north_shift'][solution]]),
                                np.array([ev_stats[ev.name][source_name +'.'+'east_shift'][solution]]))
                lat, lon = nlat[0], nlon[0]
            else:
                lat, lon = ev.lat , ev.lon

            if update_time:
                time = ev.time + ev_stats[ev.name][source_name +'.'+'time'][solution]
            else:
                time = ev.time

            tmp_ev=model.Event(lat=lat, 
                                lon=lon, 
                                time=time,
                                name=ev.name, 
                                depth=ev_stats[ev.name][source_name +'.'+'depth'][solution], 
                                elevation=None, 
                                magnitude=ev_stats[ev.name][source_name +'.'+'magnitude'][solution], 
                                moment_tensor=tmp_mt, 
                                duration=ev_stats[ev.name][source_name +'.'+'duration'][solution], 
                                tags=[f"sub-problem:{source_name}",f"frequency:{str(ev_stats[ev.name][source_name +'.'+'frequency'][solution])}" ] 
                                )            
            # Add event to dict events
            events_results.append(tmp_ev)
    else:
        print('WARNING: missing report dir for event:',ev.name)

events_results.sort(key=lambda x: x.time, reverse=False)
print('\n TOTAL EVENTS FOR CONFIG',inversions,' AND SOURCES',source_prefixes, ':', len(events_results))

if relocate:
    new_catalogue_path=os.path.join(catdir,new_catalogue_name+'_reloc_'+solution+'.pf')   
    model.dump_events(events_results, new_catalogue_path)
    print(f'\nCatalogue with relocated events saved to: {new_catalogue_path}')
else:
    new_catalogue_path=os.path.join(catdir,new_catalogue_name+'_'+solution+'.pf')   
    model.dump_events(events_results, new_catalogue_path)
    print(f'\nCatalogue with original locations saved to: {new_catalogue_path}')


 TOTAL EVENTS FOR CONFIG ['cmt_composite_LF_std_']  AND SOURCES ['vlp'] : 19

Catalogue with relocated events saved to: ../CAT/catalogue_flegrei_composite_MT_LF_std_reloc_best.pf
